# TFT v2 Full Run + Recalibration — Kaggle GPU (single session, well under 25h)
Trains the **v2** Temporal Fusion Transformer (adds `Driver` + `EngineMaker` static categoricals), evaluates it, runs the **conformal recalibration**, and bundles every artifact for download.

**Before running:**
1. Notebook settings (right panel): **Accelerator = GPU T4** (NOT P100 — cu128 torch has no sm_60 kernels), **Internet = ON**, **Persistence = Files only** (optional).
2. **+ Add Input** → attach the dataset containing the `laps_*_r*.parquet` files (same one used for run 04 — `tft_full_data.zip`). The copy cell auto-finds them.
3. Make sure the GitHub repo is pushed with the latest `main` (this notebook pulls it).

**Expected wall time:** ~2-4 h total (train ~1.5-3 h on T4 with EarlyStopping, eval + recalibration ~30 min on CPU/GPU).

**Bars to beat (v1 TFT):** green MAE **1.12s val / 1.62s test**. If v2 loses to v1, keep v1 — say so in the model card. Either result is reportable.

**When done, download `tft_v2_artifacts.zip` from the Output panel and unzip into the local repo root** (it contains `models/` + `reports/lap_time/`).

In [ ]:
# 1. Install pinned triangle on top of Kaggle's stock cu128 torch. T4 only.
!pip -q install 'pytorch-forecasting==1.7.0' 'lightning==2.6.5' mlflow fastf1 pandera scipy
import torch, pytorch_forecasting as pf, lightning
print('pf', pf.__version__, '| lightning', lightning.__version__, '| torch', torch.__version__)
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0) in {(7,5),(8,0),(8,6),(9,0)}, \
    'Switch Accelerator to T4 — this GPU arch is not in the torch build.'

In [ ]:
# 2. Clone repo (or pull) + path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
!cd $REPO && git log --oneline -1

In [ ]:
# 3. Copy parquets from the attached dataset into data/raw
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data zip as a Dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
print(len(copied), 'files | seasons:', sorted({n.split('_')[1] for n in copied}))

In [ ]:
# 4. Sanity: v2 static categoricals must include Driver + EngineMaker
from src.models.lap_time.train_tft import STATIC_CATEGORICALS
print(STATIC_CATEGORICALS)
assert 'Driver' in STATIC_CATEGORICALS and 'EngineMaker' in STATIC_CATEGORICALS, \
    'Repo not on the v2 code — git pull the latest main.'

In [ ]:
# 5. Full v2 train (100 epochs cap, EarlyStopping patience 10 on val_loss).
#    Writes models/tft_lap.{ckpt,pt} + reports/lap_time/tft_{breakdown,calibration}.csv
from src.models.lap_time.train_tft import main
main(fast=False)

In [ ]:
# 6. Compare v2 vs the v1 bars BEFORE recalibrating.
import pandas as pd
bd = pd.read_csv(f'{REPO}/reports/lap_time/tft_breakdown.csv')
ov = bd[bd.scope == 'overall'][['split', 'mae_all', 'mae_green']]
print(ov.to_string(index=False))
V1 = {'val': 1.12, 'test': 1.62}   # v1 green MAE bars
for _, r in ov.iterrows():
    beat = r.mae_green < V1[r.split]
    print(f"{r.split}: v2 green {r.mae_green:.3f} vs v1 {V1[r.split]} -> {'BEATS v1' if beat else 'LOSES to v1 — keep v1, document'}")

In [ ]:
# 7. Conformal recalibration on the FRESH checkpoint (green-flag laps, era-aware).
#    Writes models/tft_calibration.json + reports/lap_time/tft_recalibration.csv
from src.models.lap_time.recalibrate import main as recal
recal()

In [ ]:
# 8. Bundle artifacts for download (Output panel). Unzip into the LOCAL repo root.
import pandas as pd, os
for fn in ('tft_breakdown.csv', 'tft_calibration.csv', 'tft_recalibration.csv'):
    p = f'{REPO}/reports/lap_time/{fn}'
    if os.path.exists(p):
        print('\n===', fn, '===')
        print(pd.read_csv(p).to_string(index=False))
!cd $REPO && zip -qr /kaggle/working/tft_v2_artifacts.zip models reports/lap_time
print('\nDownload: /kaggle/working/tft_v2_artifacts.zip')

## After downloading (LOCAL steps)
1. Unzip `tft_v2_artifacts.zip` into the repo root (overwrites `models/tft_lap.*`, `models/tft_calibration.json`, `reports/lap_time/tft_*.csv`).
2. Run `python -m src.models.lap_time.evaluate` — folds the new TFT numbers into `model_comparison.csv`.
3. Run `pytest -q` — everything must stay green.
4. Update the model card + MASTER_CONTEXT Section 8 with the v2 numbers (win or lose vs v1 — both are results).
5. Commit the report CSVs (model binaries stay gitignored).